In [1]:
import pandas as pd

# ==================================================================
# 0. Pregunta de negocio 
# ==================================================================
# ¿Cómo se comparan los precios de arriendo por comuna en Santiago,
# ajustados a UF para que sean comparables en el tiempo (2020-2023),
# y qué comunas resultan más atractivas según ese precio ajustado?

df = pd.read_csv('../data/clean_alquiler_02_11_2023cc.csv')

## 1. Carga y vistazo general del dataset

In [2]:
df.head() # Vista general

,Unnamed: 0,id,link,titulo,precio,direction,superficie_total,superficie_util,superficie_terraza,ambientes,...,numero_piso_unidad,codigo,fecha,published_time,latitude,longitude,comuna,published,divisa,region
0,0,1,https://www.portalinmobiliario.com/mlc-1321634...,edificio av. departamental- vista norte piso 16,400000.0,"av. departamental 900 - 1200, lo vial, san miguel",34.0,34.0,0.0,3,...,16,459500,2023-02-11 00:00:00.000000,publicado hace 2 días por assetplan chile,-33.504288,-70.650536,San miguel,2023-02-10 17:56:38.667783,pesos,Rm (metropolitana)
1,1,2,https://www.portalinmobiliario.com/mlc-1321564...,edificio garcía reyes- vista poniente piso 5,410000.0,"garcía reyes 1 - 300, barrio yungay, santiago",5438.0,52.0,2.0,3,...,5,329093,2023-02-11 00:00:00.000000,publicado hace 2 días por assetplan chile,-33.444885,-70.670891,Santiago,2023-02-10 17:56:38.667831,pesos,Rm (metropolitana)
2,2,3,https://www.portalinmobiliario.com/mlc-1321897...,edificio arquería- vista norponiente piso 3,1950000.0,"arquería 1200 - 1500, rotonda atenas, las condes",190.0,140.0,50.0,5,...,3,458633,2023-02-11 00:00:00.000000,publicado hace 2 días por assetplan chile,-33.420437,-70.563446,Las condes,2023-02-10 17:56:38.667839,pesos,Rm (metropolitana)
3,3,4,https://www.portalinmobiliario.com/mlc-1318618...,edificio borgetto- vista norponiente piso 1,305500.0,"radal 1200 - 1500, blanqueado, quinta normal",389.0,32.0,62.0,2,...,1,217301,2023-02-11 00:00:00.000000,publicado hace 4 días por assetplan chile,-33.437840,-70.703300,Quinta normal,2023-02-08 17:56:38.667845,pesos,Rm (metropolitana)
4,4,5,https://www.portalinmobiliario.com/mlc-1321526...,edificio carmen- vista sur piso 2,365000.0,"carmen 1 - 300, santa isabel, santiago",38.0,33.0,5.0,2,...,2,458745,2023-02-11 00:00:00.000000,publicado hace 2 días por assetplan chile,-33.444862,-70.642990,Santiago,2023-02-10 17:56:38.667851,pesos,Rm (metropolitana)


In [3]:
print('Número de filas y columnas:', df.shape)
print('Columnas:',df.columns.tolist())
print('Tipos de dato:\n', df.dtypes)

Número de filas y columnas: (1944, 30)
Columnas: ['Unnamed: 0', 'id', 'link', 'titulo', 'precio', 'direction', 'superficie_total', 'superficie_util', 'superficie_terraza', 'ambientes', 'dormitorios', 'banos', 'estacionamientos', 'cant_max_habitantes', 'bodegas', 'gastos_comunes', 'orientacion', 'tipo_departamento', 'cantidad_pisos', 'departamentos_piso', 'numero_piso_unidad', 'codigo', 'fecha', 'published_time', 'latitude', 'longitude', 'comuna', 'published', 'divisa', 'region']
Tipos de dato:
 Unnamed: 0               int64
id                       int64
link                       str
titulo                     str
precio                 float64
direction                  str
superficie_total       float64
superficie_util        float64
superficie_terraza     float64
ambientes                int64
dormitorios              int64
banos                    int64
estacionamientos       float64
cant_max_habitantes      int64
bodegas                float64
gastos_comunes         float64
orie

In [4]:
# Muestra solo las columnas que tienen 1 o más nulos
df.isnull().sum()[df.isnull().sum() > 0]

titulo                 39
estacionamientos     1355
bodegas              1451
orientacion          1944
tipo_departamento    1021
codigo                787
latitude              390
longitude             390
comuna                 75
region                 75
dtype: int64

## 2. Columnas de interés

### 2.1 Columna `divisa` y `precio`

In [5]:
# Observar tipo de divisas existente en el dataset
df['divisa'].value_counts() 

divisa
pesos                   1694
undefined                175
undefinedconcentavos      75
Name: count, dtype: int64

In [6]:
# Observar relación entre precio y divisas no reconocidas
df[df['divisa'] != 'pesos'][['precio', 'divisa']].head(15) 

,precio,divisa
150,35.0,undefined
151,55.0,undefined
168,65.0,undefined
435,1570.0,undefinedconcentavos
436,976.0,undefinedconcentavos
437,1770.0,undefinedconcentavos
438,1912.0,undefinedconcentavos
439,61.0,undefined
440,10.0,undefined
442,836.0,undefinedconcentavos


In [7]:
# Observar estadísticas de 'precio' dentro de 'undefined', para detectar outliers
df[df['divisa'] == 'undefined']['precio'].describe() 

count     175.000000
mean       99.377143
std       678.716889
min         6.000000
25%        18.000000
50%        32.000000
75%        48.000000
max      8600.000000
Name: precio, dtype: float64

In [8]:
# Cruzar los outliers (>200) con superficie_util y comuna, para decidir si son datos reales o errores
df[(df['divisa'] == 'undefined') & (df['precio'] > 200)][['precio', 'superficie_util', 'comuna']] 

,precio,superficie_util,comuna
443,2763.0,3257.0,NaN
1385,8600.0,12.0,Las condes


In [9]:
# Confirmar si es un dato erroneo
df.loc[df['precio'] == 2763, ['titulo', 'tipo_departamento', 'dormitorios', 'banos', 'ambientes', 'superficie_util']]

,titulo,tipo_departamento,dormitorios,banos,ambientes,superficie_util
443,NaN,NaN,0,0,0,3257.0


In [10]:
# Etiquetar divisas en nueva columna
df['divisa_limpia'] = 'pesos'
df.loc[df['divisa'].isin(['undefined', 'undefinedconcentavos']), 'divisa_limpia'] = 'UF'
df.loc[df['divisa'] == 'undefinedconcentavos', 'precio'] = df.loc[df['divisa'] == 'undefinedconcentavos', 'precio'] / 100

In [11]:
# Eliminar los 2 outliers detectados arriba (2763UF y 8600UF): el resto de 'undefined' cae en el
# rango típico de precios UF (6-48), así que 200 es un corte seguro por encima de ese rango
# y por debajo de los outliers, sin tocar datos legítimos
df = df[~((df['divisa'] == 'undefined') & (df['precio'] > 200))]

In [12]:
# Comprobar si la conversion es correcta
df[df['divisa'] == 'undefinedconcentavos']['precio'].describe()

count    75.000000
mean     15.565467
std       9.514076
min       7.500000
25%       9.980000
50%      12.600000
75%      18.550000
max      69.800000
Name: precio, dtype: float64

In [13]:
# Verificación final de la columna divisa
df['divisa_limpia'].value_counts()

divisa_limpia
pesos    1694
UF        248
Name: count, dtype: int64

#### Resumen - `divisa` y `precio`

La columna traía 3 valores en vez de 2 (pesos/UF). `pesos` y `undefined` (UF) estaban bien,
solo mal etiquetados. `undefinedconcentavos` (75 filas) resultó ser también UF, con la coma
decimal perdida al parsear — se corrige dividiendo por 100. 2 outliers de `undefined`
(precio > 200) se excluyeron por no calzar con ningún patrón. <br> 
Resultado final: pesos 1694, UF 248.

### 2.2 Columna `comuna` 

In [14]:
# Ver comunas únicas
print(df['comuna'].nunique())
sorted(df['comuna'].dropna().unique())

50


['Algarrobo',
 'Antofagasta',
 'Arica',
 'Chiguayante',
 'Chillán',
 'Colina',
 'Concepción',
 'Conchalí',
 'Concón',
 'Estación central',
 'Huechuraba',
 'Independencia',
 'Iquique',
 'La cisterna',
 'La florida',
 'La reina',
 'La serena',
 'Las condes',
 'Lo barnechea',
 'Lo prado',
 'Macul',
 'Maipú',
 'Osorno',
 'Pedro aguirre cerda',
 'Peñalolén',
 'Providencia',
 'Pucón',
 'Pudahuel',
 'Puente alto',
 'Puerto montt',
 'Puerto varas',
 'Quilicura',
 'Quilpué',
 'Quinta normal',
 'Rancagua',
 'Recoleta',
 'Rm (metropolitana)',
 'San bernardo',
 'San joaquín',
 'San miguel',
 'San pedro de la paz',
 'San ramón',
 'Santiago',
 'Temuco',
 'Valdivia',
 'Valparaíso',
 'Villa alemana',
 'Vitacura',
 'Viña del mar',
 'Ñuñoa']

In [15]:
# Rellenar comuna nula extrayéndola de 'direction' (dirección -> sector -> comuna)
df.loc[df['comuna'].isnull(), 'comuna'] = (
    df.loc[df['comuna'].isnull(), 'direction']
    .str.split(',')
    .str[-1]
    .str.strip()
)

In [16]:
# Normalizar a Title Case
df['comuna'] = df['comuna'].str.strip().str.lower().str.title()

# Corregir las preposiciones
preposiciones = [' De ', ' Del ', ' La ', ' Las ', ' Los ', ' Y ']
for prep in preposiciones:
    df['comuna'] = df['comuna'].str.replace(prep, prep.lower(), regex=False)

In [17]:
sorted(df['comuna'].unique())
df['comuna'].nunique()

50

In [18]:
sorted(df['comuna'].unique())

['Algarrobo',
 'Antofagasta',
 'Arica',
 'Chiguayante',
 'Chillán',
 'Colina',
 'Concepción',
 'Conchalí',
 'Concón',
 'Estación Central',
 'Huechuraba',
 'Independencia',
 'Iquique',
 'La Cisterna',
 'La Florida',
 'La Reina',
 'La Serena',
 'Las Condes',
 'Lo Barnechea',
 'Lo Prado',
 'Macul',
 'Maipú',
 'Osorno',
 'Pedro Aguirre Cerda',
 'Peñalolén',
 'Providencia',
 'Pucón',
 'Pudahuel',
 'Puente Alto',
 'Puerto Montt',
 'Puerto Varas',
 'Quilicura',
 'Quilpué',
 'Quinta Normal',
 'Rancagua',
 'Recoleta',
 'Rm (Metropolitana)',
 'San Bernardo',
 'San Joaquín',
 'San Miguel',
 'San Pedro de la Paz',
 'San Ramón',
 'Santiago',
 'Temuco',
 'Valdivia',
 'Valparaíso',
 'Villa Alemana',
 'Vitacura',
 'Viña del Mar',
 'Ñuñoa']

In [19]:
# Cuántas filas quedaron con el valor equivocado
(df['comuna'] == 'Rm (Metropolitana)').sum()

np.int64(1)

In [20]:
# Ver el texto de dirección original de esas filas, para entender por qué falló el split
df[df['comuna'] == 'Rm (Metropolitana)'][['direction', 'comuna']]

,direction,comuna
290,"sta clara 300 - 600, lo ovalle, la cisterna",Rm (Metropolitana)


In [21]:
# Corregir la única fila con la comuna mal etiquetada (traía el valor de 'region' en vez de la comuna real)
df.loc[df['comuna'] == 'Rm (Metropolitana)', 'comuna'] = 'La Cisterna'

In [22]:
df['region'].value_counts()

region
Rm (metropolitana)         1760
Valparaíso                   62
Biobío                       16
La araucanía                  6
Libertador b. o'higgins       5
Propiedades usadas            4
Los lagos                     4
Coquimbo                      4
Arica y parinacota            2
Tarapacá                      2
Antofagasta                   1
Los ríos                      1
Ñuble                         1
Name: count, dtype: int64

In [23]:
# Acotar el dataset a la Región Metropolitana, según el alcance definido para la pregunta de negocio
antes = len(df)
df = df[df['region'] == 'Rm (metropolitana)']
print(f'Filas antes del filtro: {antes} -> después: {len(df)}')

Filas antes del filtro: 1942 -> después: 1760


#### Resumen - `comuna`

Se rellenaron los valores nulos de `comuna` a partir de la columna `direction`, y se corrigió
un dato que tenía por comuna el nombre de la región (RM). Además, se acotó el dataset a la
Región Metropolitana, ya que incluía comunas de otras regiones (Viña del Mar, Concepción,
entre otras) fuera del alcance de la pregunta de negocio. <br>
Resultado final: filas antes del filtro: 1942 → después: 1760

### 2.3 Columna `published`

In [24]:
# Convertir str a datetime
df['published'] = pd.to_datetime(df['published'])

In [25]:
# Quedarse solo con la fecha (sin hora)
df['fecha_publicacion'] = df['published'].dt.date